# 使用Baichuan-7B对人物分类数据进行微调

## 1. 数据集
通过大模型提示词使用大模型标注原始人物之间的关系，如下所示
```
{
  "conversation_id": 1,
  "category": "relation classification",
  "conversation": [
    {
      "human": "给定以下标签：['不确定', '夫妻', '父母', '兄弟姐妹', '上下级', '师生', '好友', '同学', '合作', '同一个人', '情侣', '祖孙', '同门', '亲戚']，请在以下句子中分析并分类实体之间的关系：'与李源澄论戴东原书'在这个句子中，戴东原和李源澄之间的关系应该属于哪个标签？",
      "assistant": "不知道"
    }
  ]
}
```

## 2. 模型微调
使用Firefly框架进行模型微调，访问网址为：https://github.com/yangjianxin1/Firefly.本文基于Baichuan-7b为基座模型，采用QLora方式训练，训练参数如下：
```
{
    "output_dir": "output/firefly-baichuan-7b-people",
    "model_name_or_path": "/home/test/baichun_7b",
    "train_file": "./data/train.jsonl",
    "num_train_epochs": 5,
    "per_device_train_batch_size": 8,
    "gradient_accumulation_steps": 2,
    "learning_rate": 2e-4,
    "max_seq_length": 256,
    "logging_steps": 100,
    "save_steps": 100,
    "save_total_limit": 1,
    "lr_scheduler_type": "constant_with_warmup",
    "warmup_steps": 100,
    "lora_rank": 64,
    "lora_alpha": 16,
    "lora_dropout": 0.05,

    "gradient_checkpointing": true,
    "disable_tqdm": false,
    "optim": "paged_adamw_32bit",
    "seed": 42,
    "fp16": true,
    "report_to": "tensorboard",
    "dataloader_num_workers": 0,
    "save_strategy": "steps",
    "weight_decay": 0,
    "max_grad_norm": 0.3,
    "remove_unused_columns": false
}
```
使用命令行torchrun --nproc_per_node=2 train_qlora.py --train_args_file train_args/qlora/baichuan-7b-sft-qlora.json进行训练，训练时间大约20分钟，最终的train loss为0.0273。
  在Firefly框架中设置好merge_lora.py中的模型文件路径，将adapter的权重与Baichuan-7b模型合并，合并得到新文件firefly-baichuan-7b-people-merge。
  在Firefly框架，仿造script/chat/single_chat.py文件，将其改写成API调用方式的文件single_chat_server.py，代码如下：

In [ ]:
# -*- coding: utf-8 -*-
# @place: Pudong, Shanghai
# @file: single_chat_server.py
# @time: 2023/7/25 22:27
from transformers import AutoModelForCausalLM, AutoTokenizer
import torch
# 单轮对话web服务
from flask import Flask, request, jsonify

app = Flask("single_chat_server")


@app.route('/people_rel_cls', methods=['POST'])
def predict():
    req_dict = request.json
    text, people1, people2 = req_dict["text"], req_dict["people1"], req_dict["people2"]
    text = text.strip()
    content = f"给定以下标签：['不确定', '夫妻', '父母', '兄弟姐妹', '上下级', '师生', '好友', '同学', " \
              f"'合作', '同一个人', '情侣', '祖孙', '同门', '亲戚']，" \
              f"请在以下句子中分析并分类实体之间的关系：'{text}'" \
              f"在这个句子中，{people1}和{people2}之间的关系应该属于哪个标签？"
    print(content)
    input_ids = tokenizer(content, return_tensors="pt",
                          add_special_tokens=False).input_ids.to(device)
    with torch.no_grad():
        outputs = model.generate(
            input_ids=input_ids, max_new_tokens=max_new_tokens, do_sample=True,
            top_p=top_p, temperature=temperature, repetition_penalty=repetition_penalty,
            eos_token_id=tokenizer.eos_token_id
        )
    outputs = outputs.tolist()[0][len(input_ids[0]):]
    response = tokenizer.decode(outputs)
    print(outputs, response)
    response = response.strip().replace(text, "").replace(
        '</s>', "").replace('<s>', "").strip()
    return jsonify({"result": response})


if __name__ == '__main__':
    model_name = "/home/test/Firefly/script/checkpoint/firefly-baichuan-7b-people-merge"
    max_new_tokens = 5
    top_p = 0.9
    temperature = 0.01
    repetition_penalty = 1.0
    device = 'cuda:0'
    input_pattern = '<s>{}</s>'
    model = AutoModelForCausalLM.from_pretrained(
        model_name,
        trust_remote_code=True,
        low_cpu_mem_usage=True,
        torch_dtype=torch.float16,
    ).to(device).eval()
    tokenizer = AutoTokenizer.from_pretrained(
        model_name,
        trust_remote_code=True,
        # llama不支持fast
        use_fast=False if model.config.model_type == 'llama' else True
    )
    print("model loaded!")
    app.run(host="0.0.0.0", port=5000, threaded=True)